In [ ]:
from lime import lime_image
from transformers import AutoFeatureExtractor, AutoModel
import torch
import pickle
import numpy as np
from PIL import Image
import pandas as pd

In [ ]:
device = "cuda"
feature_extractor = AutoFeatureExtractor.from_pretrained("google/vit-base-patch16-224")
model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(device)
model.eval()
@torch.no_grad()
def extract_feature(image, pool_index=0):
    """Genera le feature facendo CLS-pooling"""
    inputs = feature_extractor(image, return_tensors="pt")
    for k in inputs:
        inputs[k] = inputs[k].to(device)
    outputs = model(**inputs)
    pooled_output = outputs.last_hidden_state[:, pool_index].reshape(-1)
    model.zero_grad()
    del outputs
    del inputs
    return pooled_output

In [ ]:
path_images = "../input/bigImageDataset/imgs"

id2img = pickle.load(open("../embeddings/unique2path.pkl", "rb"))


def openImage(image):
    path=f'{path_images}/{image}'
    return Image.open(path).convert('RGBA').convert('RGB')

In [ ]:
def merge_images(image1,image2,fixed_size=(512,512)):
    image1 = image1.resize((fixed_size[0], fixed_size[1]))
    image2 = image2.resize((fixed_size[0], fixed_size[1]))
    new_image = Image.new("RGB", (fixed_size[0]*2, fixed_size[1]), "black")
    new_image.paste(image1,(0,0))
    new_image.paste(image2,(fixed_size[1],0))
    return new_image

def unmerge_images(image,fixed_size=(512,512)):
    image1=image.crop((0, 0, fixed_size[0], fixed_size[1]))
    image2=image.crop((fixed_size[0], 0, fixed_size[0]*2, fixed_size[1]))
    return image1,image2

In [ ]:
images=list(id2img.items())[:2]
images

In [ ]:
image1=openImage(images[0][1])
image2=openImage(images[1][1])
merged=merge_images(image1,image2)
merged

In [ ]:
SOGLIA=0.5
def batch_predictor_image(images):
    images=[Image.fromarray(np.uint8(i)) for i in images]
    d=torch.nn.CosineSimilarity()
    distances=[]
    with torch.no_grad():
        for images_merged in images:
            image_src,image_trg=unmerge_images(images_merged)
            emb_src=extract_feature(image_src)
            emb_trg=extract_feature(image_trg)
            distance=d(emb_src.unsqueeze(0),emb_trg.unsqueeze(0)).item()
            distance_neg=1-distance
            distance_pos=distance
            distances.append([distance_neg,distance_pos])
    distances=np.array(distances).reshape(-1,2)
    # distances= np.exp(distances)
    # p=torch.from_numpy(distances)
    # distances=torch.nn.functional.softmax(p,1).detach().cpu().numpy()
    return distances


In [ ]:
graph_path="../graphsV2/NFTLevel-Pruning_V2-Weekly2/(2021, 3)/edge_list.csv"
edge_list=pd.read_csv(graph_path)
edge_list=edge_list[edge_list.weight<.74].sort_values(by="weight",ascending=False)
edge_list=edge_list.reset_index(drop=True)
edge_list.head(10)

In [ ]:
edge_index=3
edge=edge_list.iloc[edge_index]
print(edge)
image1=openImage(id2img[edge["source"]])
image2=openImage(id2img[edge["target"]])
merged=merge_images(image1,image2)
merged

In [ ]:
print("Logits:",batch_predictor_image([merged]))


In [ ]:
explainer_image = lime_image.LimeImageExplainer(kernel_width=0.5,feature_selection="lasso_path")
explaination_image= explainer_image.explain_instance(
    np.array(merged),
    batch_predictor_image,
    labels=["Not similar","Similar"],
    top_labels=2,
    hide_color=1,
    num_samples=10000,
    num_features=100000000,
    batch_size=100
)

In [ ]:
import matplotlib.pyplot as plt
from skimage.segmentation import mark_boundaries
fig,ax=plt.subplots(2,1,figsize=(30,15))
temp, mask = explaination_image.get_image_and_mask(
    1, 
    positive_only=False, 
    num_features=50,
    hide_rest=False)
img_boundry2 = mark_boundaries(temp/255.0, mask)
ax[0].imshow(merged)
ax[1].imshow(img_boundry2)
plt.show()